# 01 - VAE Exploration

本 notebook 演示 VAE (Variational Autoencoder) 的训练和可视化。

VAE 将 28x28 的 MNIST 图像压缩到 4x4 的 latent space，作为后续扩散模型的工作空间。

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from models.vae import VAE

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. 加载 MNIST 数据集

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 查看一些样本
images, labels = next(iter(loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i, 0] * 0.5 + 0.5, cmap='gray')
    ax.set_title(f'Label: {labels[i].item()}')
    ax.axis('off')
plt.suptitle('MNIST Samples')
plt.show()

## 2. 训练 VAE

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

vae = VAE(latent_dim=1024).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

# 训练 20 个 epoch
losses = []
for epoch in range(20):
    vae.train()
    total_loss = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/20')
    for x, _ in pbar:
        x = x.to(device)
        recon, mu, logvar = vae(x)
        loss, recon_loss, kl_loss = vae.loss(x, recon, mu, logvar)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item() / x.size(0))
    
    avg_loss = total_loss / len(dataset)
    losses.append(avg_loss)
    print(f'Epoch {epoch+1}: Loss = {avg_loss:.4f}')

## 3. 训练损失曲线

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 4. 重建效果展示

In [ ]:
vae.eval()
with torch.no_grad():
    images, _ = next(iter(loader))
    images = images[:16].to(device)
    recon, _, _ = vae(images)

# 对比原图和重建图
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i in range(8):
    # 原图
    axes[0, i].imshow(images[i, 0].cpu() * 0.5 + 0.5, cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=12)
    
    # 重建图
    axes[1, i].imshow(recon[i, 0].cpu() * 0.5 + 0.5, cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Reconstructed', fontsize=12)
    
    # 原图
    axes[2, i].imshow(images[i+8, 0].cpu() * 0.5 + 0.5, cmap='gray')
    axes[2, i].axis('off')
    
    # 重建图
    axes[3, i].imshow(recon[i+8, 0].cpu() * 0.5 + 0.5, cmap='gray')
    axes[3, i].axis('off')

plt.suptitle('VAE Reconstruction Results', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Latent Space 采样

从 latent space 随机采样，生成新的数字图像。

In [ ]:
# 从标准正态分布采样
z = torch.randn(16, 1024).to(device)
with torch.no_grad():
    generated = vae.decode(z)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i, 0].cpu() * 0.5 + 0.5, cmap='gray')
    ax.axis('off')
plt.suptitle('Generated from Random Latent Vectors', fontsize=14)
plt.show()

## 6. Latent Space 插值

在两个数字的 latent vector 之间插值，观察形态的渐变过程。

In [ ]:
# 获取两个不同数字的 latent vector
vae.eval()
with torch.no_grad():
    # 找一个 '0' 和一个 '1'
    idx_0 = (labels == 0).nonzero(as_tuple=True)[0][0]
    idx_1 = (labels == 1).nonzero(as_tuple=True)[0][0]
    
    z_0, _, _ = vae.encode(images[idx_0:idx_0+1].to(device))
    z_1, _, _ = vae.encode(images[idx_1:idx_1+1].to(device))
    
    # 线性插值
    n_steps = 10
    alphas = torch.linspace(0, 1, n_steps).to(device)
    z_interp = torch.stack([z_0 * (1-a) + z_1 * a for a in alphas]).squeeze(1)
    
    # 解码
    decoded = vae.decode(z_interp)

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2))
for i, ax in enumerate(axes):
    ax.imshow(decoded[i, 0].cpu() * 0.5 + 0.5, cmap='gray')
    ax.set_title(f'a={alphas[i]:.1f}')
    ax.axis('off')
plt.suptitle('Latent Space Interpolation (0 → 1)', fontsize=14)
plt.show()

## 7. 保存模型

训练完成后保存 VAE，供后续扩散模型使用。

In [ ]:
import os
os.makedirs('../checkpoints', exist_ok=True)
torch.save(vae.state_dict(), '../checkpoints/vae.pt')
print('VAE saved to ../checkpoints/vae.pt')

## 总结

VAE 学会了将 MNIST 图像压缩到低维 latent space，并能从中重建和生成新的图像。

下一步: 使用 U-Net 在这个 latent space 上进行扩散模型训练。